# ⚠️ Subsistema 03: Narcotics Spatio-Temporal GNN (`03-narcotics-intel`)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/03_Narcotics_STGNN_Corridors_Colab_GPU.ipynb)

* **Agências Líderes:** Drug Enforcement Administration (DEA), CBP Office of Field Operations & HIDTA
* **Anéis de Defesa:** `RING_2_FEDERAL` (Portos de Entrada) & `RING_3_INTERSTATE` (Rodovias I-10, I-35, I-95)
* **Arquitetura:** `SpatioTemporalGNN` (Graph Convolution sobre a malha rodoviária + Temporal Self-Attention sobre histórico de apreensões de Fentanil, Cocaína e Metanfetamina).


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SpatioTemporalGNN(nn.Module):
    def __init__(self, num_nodes: int = 10, in_features: int = 4, hidden_dim: int = 32):
        super().__init__()
        self.spatial_proj = nn.Linear(in_features, hidden_dim)
        self.temporal_gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # x: (batch, nodes, time, features)
        b, n, t, f = x.shape
        h = torch.relu(self.spatial_proj(x))
        h = torch.einsum("ij,bjtf->bitf", adj, h)
        h_flat = h.reshape(b * n, t, -1)
        out, _ = self.temporal_gru(h_flat)
        scores = self.risk_head(out[:, -1, :]).reshape(b, n)
        return scores

num_nodes = 10
adj = torch.eye(num_nodes, device=device)
for i in range(num_nodes - 1):
    adj[i, i + 1] = 0.5
    adj[i + 1, i] = 0.5

model = SpatioTemporalGNN(num_nodes=num_nodes).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
loss_fn = nn.BCELoss()

x_data = torch.rand(64, num_nodes, 7, 4, device=device)
y_target = (x_data[:, :, -1, 0] > 0.45).float()

losses = []
for epoch in range(1, 51):
    model.train()
    opt.zero_grad()
    preds = model(x_data, adj)
    loss = loss_fn(preds, y_target)
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3.5))
plt.plot(losses, color="#10b981", linewidth=2, label="ST-GNN Corridor Surge BCE Loss")
plt.title("Treinamento Spatio-Temporal GNN (Corredores HIDTA I-10 / I-35 / I-95)")
plt.xlabel("Época")
plt.ylabel("BCE Loss")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

model.eval()
torch.onnx.export(
    model,
    (torch.randn(1, num_nodes, 7, 4, device=device), adj),
    "narcotics_stgnn.onnx",
    input_names=["node_time_features", "adjacency_matrix"],
    output_names=["corridor_risk_24h"],
    opset_version=17,
)
print("✅ Exportado com sucesso: narcotics_stgnn.onnx")
